# 01 · Baseline: TF-IDF + Logistic Regression

Classical baseline for **Banking77** (77 customer-support intents).

* Features: TF-IDF over word **1–2 grams**
* Classifier: multinomial **Logistic Regression**
* Metrics on the official **test** split: accuracy and **macro-F1**. Macro-F1 averages F1 over the 77 intents, so small intents count as much as large ones.
* Output: the **10 most-confused intent pairs**

Artifacts written to `results/`:

| File | Content |
|---|---|
| `baseline_metrics.json` | test accuracy / macro-F1 (read by notebook 02) |
| `baseline_classification_report.csv` | per-intent precision / recall / F1 |
| `baseline_confused_pairs.csv` | top-10 confused intent pairs |

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import ClassLabel, load_dataset
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

SEED = 42
# Resolve the project root whether the notebook runs from notebooks/, the repo root or Colab (/content).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
BANKING77_REVISION = "90d4e2ee5521c04fc1488f065b8b083658768c57"  # pinned Hub commit of PolyAI/banking77
BANKING77_CSV = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"


def load_banking77():
    """Load PolyAI/banking77 as a DatasetDict(train, test) with a 77-way ClassLabel `label` column.

    The Hub repo "PolyAI/banking77" is a legacy loading script, which `datasets>=4` no longer
    executes. We load the exact CSVs that script downloads and take the canonical label order
    from the repo's dataset_infos.json, so label ids match the Hub dataset.
    """
    info_path = hf_hub_download("PolyAI/banking77", "dataset_infos.json", repo_type="dataset",
                                revision=BANKING77_REVISION)
    info = json.loads(Path(info_path).read_text(encoding="utf-8"))
    label_feature = ClassLabel(names=info["default"]["features"]["label"]["names"])
    ds = load_dataset("csv", data_files={"train": BANKING77_CSV + "train.csv", "test": BANKING77_CSV + "test.csv"})
    # Map intent names to ids with the canonical order (str2int raises on unknown names).
    ds = ds.map(lambda b: {"label": label_feature.str2int(b["category"])}, batched=True, remove_columns=["category"])
    return ds.cast_column("label", label_feature)

D:\Projects\IntentRoute\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the data

In [2]:
ds = load_banking77()
label_names = ds["train"].features["label"].names
print(ds)
print(f"{len(label_names)} intents")

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()
train_df.sample(5, random_state=SEED).assign(intent=lambda d: d["label"].map(label_names.__getitem__))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
77 intents


,text,label,intent
6883,Is it possible for me to change my PIN number?,21,change_pin
5836,I'm not sure why my card didn't work,25,declined_card_payment
8601,I don't think my top up worked,59,top_up_failed
2545,Can you explain why my payment was charged a fee?,15,card_payment_fee_charged
8697,How long does a transfer from a UK account tak...,5,balance_not_updated_after_bank_transfer


In [3]:
# Class balance on the train split: the classes are imbalanced, so we report macro-F1 as well as accuracy.
counts = train_df["label"].value_counts()
print(f"Train examples per intent: min={counts.min()}, median={int(counts.median())}, max={counts.max()}")
print(f"Test examples per intent : {test_df['label'].value_counts().unique()}")

Train examples per intent: min=35, median=127, max=187
Test examples per intent : [40]


## 2. Train

The only hyperparameter we tune is the regularisation strength `C`. We pick it with 5-fold stratified cross-validation **on the train split only**, then refit on the full train split. The test split is used once, for the final evaluation.

In [4]:
pipeline = Pipeline([
    # Word unigrams + bigrams; sublinear TF dampens repeated tokens in longer queries.
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000)),
])

search = GridSearchCV(
    pipeline,
    param_grid={"clf__C": [1.0, 10.0, 100.0]},
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=2,             # each 77-class LBFGS fit needs a few hundred MB; unbounded parallelism can run out of RAM
    error_score="raise",  # fail loudly instead of silently scoring failed fits as NaN
)
search.fit(train_df["text"], train_df["label"])
model = search.best_estimator_  # refit on the full train split
print(f"Best C: {search.best_params_['clf__C']}  |  mean CV macro-F1 (train folds): {search.best_score_:.4f}")

Best C: 10.0  |  mean CV macro-F1 (train folds): 0.8768


## 3. Evaluate on the test split

In [5]:
y_true = test_df["label"].to_numpy()
y_pred = model.predict(test_df["text"])

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Test accuracy : {accuracy:.4f}")
print(f"Test macro-F1 : {macro_f1:.4f}")

Test accuracy : 0.8938
Test macro-F1 : 0.8942


In [6]:
# Per-intent report, saved in full; the 10 weakest intents by F1 are shown here.
report = pd.DataFrame(
    classification_report(
        y_true, y_pred,
        labels=range(len(label_names)), target_names=label_names,
        output_dict=True, zero_division=0,
    )
).T
report.to_csv(RESULTS_DIR / "baseline_classification_report.csv")
report.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("f1-score").head(10).round(3)

,precision,recall,f1-score,support
balance_not_updated_after_bank_transfer,0.646,0.775,0.705,40.0
verify_my_identity,0.738,0.775,0.756,40.0
pending_transfer,0.824,0.700,0.757,40.0
card_not_working,0.673,0.925,0.779,40.0
top_up_failed,0.762,0.800,0.780,40.0
card_payment_not_recognised,0.853,0.725,0.784,40.0
wrong_exchange_rate_for_cash_withdrawal,0.833,0.750,0.789,40.0
topping_up_by_card,0.879,0.725,0.795,40.0
transfer_not_received_by_recipient,0.767,0.825,0.795,40.0
extra_charge_on_statement,0.773,0.850,0.810,40.0


## 4. Confusion analysis: the 10 most-confused intent pairs

A pair (A, B) counts errors in **both directions**: true A predicted as B, plus true B predicted as A.

In [7]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))

# Fold the matrix onto its upper triangle so each unordered pair appears once (diagonal = correct, excluded).
pair_totals = np.triu(cm + cm.T, k=1)
a_idx, b_idx = np.nonzero(pair_totals)

confused_pairs = (
    pd.DataFrame({
        "intent_a": [label_names[a] for a in a_idx],
        "intent_b": [label_names[b] for b in b_idx],
        "a_predicted_as_b": cm[a_idx, b_idx],
        "b_predicted_as_a": cm[b_idx, a_idx],
        "total_confusions": pair_totals[a_idx, b_idx],
    })
    .sort_values(["total_confusions", "intent_a", "intent_b"], ascending=[False, True, True])
    .head(10)
    .reset_index(drop=True)
)
confused_pairs.index = confused_pairs.index + 1
confused_pairs.to_csv(RESULTS_DIR / "baseline_confused_pairs.csv", index_label="rank")
confused_pairs

,intent_a,intent_b,a_predicted_as_b,b_predicted_as_a,total_confusions
1,verify_my_identity,why_verify_identity,6,6,12
2,balance_not_updated_after_bank_transfer,transfer_not_received_by_recipient,4,4,8
3,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal,2,5,7
4,top_up_failed,top_up_reverted,3,4,7
5,unable_to_verify_identity,verify_my_identity,5,2,7
6,balance_not_updated_after_bank_transfer,pending_transfer,2,4,6
7,card_acceptance,card_not_working,5,1,6
8,get_physical_card,pin_blocked,2,4,6
9,card_payment_not_recognised,extra_charge_on_statement,4,1,5
10,exchange_via_app,fiat_currency_support,1,4,5


## 5. Save metrics for the comparison in notebook 02

In [8]:
baseline_metrics = {
    "model": "TF-IDF (word 1-2 grams) + LogisticRegression",
    "split": "test",
    "n_test": int(len(y_true)),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "best_C": float(search.best_params_["clf__C"]),
}
(RESULTS_DIR / "baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2))
print(json.dumps(baseline_metrics, indent=2))

{
  "model": "TF-IDF (word 1-2 grams) + LogisticRegression",
  "split": "test",
  "n_test": 3080,
  "accuracy": 0.8938311688311689,
  "macro_f1": 0.8941886071786473,
  "best_C": 10.0
}
